# 中间件

[中间件](https://docs.langchain.com/oss/python/langchain/middleware/overview)（middleware）是本次更新中最亮眼的特性，诸多新功能均藉由中间件实现，比如人机交互、动态系统提示词、动态注入上下文等等。中间件是一种钩子函数。通过向工作流中预埋中间件，能够实现工作流的高效拓展和可定制化。

LangChain 通过 [装饰器](https://reference.langchain.com/python/langchain/middleware/#decorators) 创建 **自定义中间件**。

```{dropdown} 装饰器类型（点击展开）

  | DECORATOR | DESCRIPTION |
  | -- | -- |
  | `@before_agent` | 在 Agent 执行前执行逻辑 |
  | `@after_agent` | 在 Agent 执行后执行逻辑 |
  | `@before_model` | 在每次模型调用前执行逻辑 |
  | `@after_model` | 在每次模型收到响应后执行逻辑 |
  | `@wrap_model_call` | 控制模型的调用过程 |
  | `@wrap_tool_call` | 控制工具的调用过程 |
  | `@dynamic_prompt` | 动态生成系统提示词 |
  | `@hook_config` | 配置钩子行为 |

```

**装饰器类型** 决定中间件的执行位置。比如使用 `@before_model` 装饰器，能够在模型调用前执行特定逻辑。**被装饰函数** 负责这段特定逻辑的具体实现。这么说可能有点抽象。没关系，本节提供了四个例子，看完你一定能够领悟到中间件的使用方法：

- 预算控制
- 消息截断
- 敏感词过滤
- PII 检测`（个人隐私信息检测）`

## 一、预算控制

随着对话轮次增加，对话记录也越来越长，从而导致请求费用上升。为了控制预算，可以设定在对话轮次超过某个阈值后，切换到低费率模型。下面我们用自定义中间件实现这个功能。

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage
from langgraph.graph import MessagesState

# 加载 .env 文件中的环境变量（API Key、Base URL 等）
_ = load_dotenv()

# 低费率模型：用于长对话或简单任务，费用较低
basic_model = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),  # 从环境变量读取 API 密钥
    base_url=os.getenv("DASHSCOPE_BASE_URL"),  # 从环境变量读取 API 基础 URL
    model="qwen3-coder-plus",  # 低费率模型名称
)

# 高费率模型：用于短对话或复杂任务，能力更强但费用更高
advanced_model = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    model="qwen3-max",  # 高费率模型名称
)


由于我们的修改涉及模型推理，`@before_model` 和 `@after_model` 在这里已经不够用了。我们选用可以干涉模型调用的 [`@wrap_model_call`](https://reference.langchain.com/python/langchain/middleware/#langchain.agents.middleware.wrap_model_call) 装饰器。具体逻辑由函数 `dynamic_model_selection` 实现：当历史对话超过 5 条时，自动切换到低费率模型。

In [2]:
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """
    自定义中间件：根据对话长度动态选择模型。
    当历史消息数超过 5 条时，自动切换到低费率模型以控制预算。

    Args:
        request: ModelRequest 对象，包含当前状态（state）、消息列表等信息
        handler: 实际执行模型调用的处理函数

    Returns:
        ModelResponse: 模型返回的响应
    """
    # 获取当前对话历史中的消息总数（包括用户消息和 AI 回复）
    message_count = len(request.state["messages"])

    # 根据消息数量决定使用哪个模型：
    # 超过 5 条 → 低费率模型（控制成本）
    # 5 条及以下 → 高费率模型（保证质量）
    if message_count > 5:
        model = basic_model  # 对话变长后切换到低费率模型
    else:
        model = advanced_model  # 短对话使用高费率模型

    print(f"message_count: {message_count}")
    print(f"model_name: {model.model_name}")

    # request.override(model=model) 创建一个新的请求对象，替换其中的模型
    # handler 接收修改后的请求并执行实际的模型调用
    return handler(request.override(model=model))

# 创建 Agent，将自定义中间件注册到工作流中
agent = create_agent(
    model=advanced_model,  # 默认使用高费率模型（中间件会按需覆盖）
    middleware=[dynamic_model_selection]  # 注册预算控制中间件
)


从下面的例子可以看出，当历史对话数 `message_count` 超过 5 条时，确实从高费率模型 `qwen3-max` 切换到低费率模型 `qwen3-coder-plus`。我们成功实现了预算控制功能！

In [3]:
# 初始化对话状态，messages 列表为空
state: MessagesState = {"messages": []}
# 准备 4 个不同主题的测试输入，用于模拟多轮对话
items = ['汽车', '飞机', '摩托车', '自行车']
for idx, i in enumerate(items):
    print(f"\n=== Round {idx+1} ===")

    # 对话计数说明：用户提问 + AI 回复各算一条消息
    # 每轮 invoke 结束后 messages 增加 2 条（一问一答）
    # Round 1 开始前 0 条 → 提问后 1 条 → 回复后 2 条
    # Round 2 开始前 2 条 → 提问后 3 条 → 回复后 4 条
    # Round 3 开始前 4 条 → 提问后 5 条 → 回复后 6 条（超过阈值 5，触发模型切换）
    # Round 4 开始前 6 条 → 提问后 7 条（已超过阈值，使用低费率模型）

    # 添加用户消息到状态中
    state["messages"] += [HumanMessage(content=f"{i}有几个轮子，请简单回答")]

    # 调用 Agent，中间件会在此处根据消息数量决定是否切换模型
    result = agent.invoke(state)

    # 更新状态：将 Agent 的完整消息历史（包含 AI 回复）保存回 state
    state["messages"] = result["messages"]

    # 打印 AI 的回复内容（messages 列表的最后一条）
    print(f'content: {result["messages"][-1].content}')



=== Round 1 ===
message_count: 1
model_name: qwen3-max
content: 普通汽车通常有4个轮子。

=== Round 2 ===
message_count: 3
model_name: qwen3-max
content: 飞机轮子数量不固定，常见客机一般有6到10个轮子。

=== Round 3 ===
message_count: 5
model_name: qwen3-max
content: 摩托车通常有2个轮子。

=== Round 4 ===
message_count: 7
model_name: qwen3-coder-plus
content: 自行车通常有2个轮子。


## 二、消息截断

LLM 的上下文存在长度限制。一旦超过限制，就需要对上下文进行压缩。在众多处理方案中，消息截断是最简单的。下面我们通过 `@before_model` 装饰器实现消息截断功能。

In [4]:
from langchain.messages import RemoveMessage  # 用于从消息列表中移除指定消息
from langgraph.graph.message import REMOVE_ALL_MESSAGES  # 特殊标记，表示移除所有已有消息
from langgraph.checkpoint.memory import InMemorySaver  # 内存中的会话检查点保存器，用于持久化对话状态
from langchain.agents import create_agent, AgentState  # Agent 创建函数和状态类型
from langchain.agents.middleware import before_model  # @before_model 装饰器：在模型调用前执行
from langgraph.runtime import Runtime  # 运行时上下文对象，提供中间件执行时的环境信息
from langchain_core.runnables import RunnableConfig  # 运行配置对象，用于传递 thread_id 等参数
from typing import Any  # 通用类型注解


我们尝试一种截断策略：在保留最近消息的同时，额外保留第一条消息。在下面的例子中，由于我们在第一条消息中就告诉智能体「我是 bob」，因此它记得我是 bob.

In [5]:
@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    消息截断中间件：在模型调用前裁剪上下文窗口。

    截断策略：保留第一条消息（包含初始设定，如用户身份）和最近的几条消息。
    这样既能控制上下文长度，又不会丢失重要的初始信息。

    Args:
        state: 当前 Agent 状态，包含 messages 列表
        runtime: 运行时上下文

    Returns:
        如果消息数 <= 3，返回 None（不做任何修改）
        否则返回包含截断后消息列表的字典，用于更新 state
    """
    messages = state["messages"]

    # 消息数不超过 3 条时无需截断，直接返回 None 表示不修改 state
    if len(messages) <= 3:
        return None

    # 保留第一条消息（通常包含系统设定或用户身份等关键信息）
    first_msg = messages[0]

    # 根据消息总数的奇偶性决定保留最近的消息数量：
    # 偶数条 → 保留最近 3 条
    # 奇数条 → 保留最近 4 条
    # （这样可以保持消息的完整性，确保用户-AI 对话对不被拆分）
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]

    # 合并：第一条消息 + 最近的几条消息
    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            # RemoveMessage + REMOVE_ALL_MESSAGES：先清空所有已有消息
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            # 再插入截断后的新消息列表
            *new_messages
        ]
    }

# 创建 Agent 并注册消息截断中间件
agent = create_agent(
    basic_model,  # 使用低费率模型
    middleware=[trim_messages],  # 注册截断中间件
    checkpointer=InMemorySaver(),  # 启用内存检查点，使多轮对话能保持会话状态
)

# 配置：使用 thread_id 标识会话线程，配合 checkpointer 实现状态持久化
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

def agent_invoke(agent):
    """连续调用 Agent 4 轮，最后打印 AI 的回复。"""
    # 第 1 轮：用户介绍自己的名字
    agent.invoke({"messages": "hi, my name is bob"}, config)
    # 第 2 轮：要求写一首关于猫的诗
    agent.invoke({"messages": "write a short poem about cats"}, config)
    # 第 3 轮：要求写一首关于狗的诗
    agent.invoke({"messages": "now do the same but for dogs"}, config)
    # 第 4 轮：询问用户名字，检验截断中间件是否保留了第一条消息中的信息
    final_response = agent.invoke({"messages": "what's my name?"}, config)

    # 打印最终 AI 回复
    final_response["messages"][-1].pretty_print()

# 执行测试
agent_invoke(agent)


================================== Ai Message ==================================

Your name is Bob! You told me "hi, my name is bob" at the beginning of our conversation.


当然，这个表现不足以说明截断中间件真的生效了。若这个中间件从未生效，也会有这样的结果。为了证明它真的生效了，我们再次修改截断策略。这次只保留最后两条对话记录。如果智能体不记得我是 bob，说明截断中间件确实起作用了。

In [6]:
@before_model
def trim_without_first_message(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    激进的消息截断中间件：仅保留最后两条消息。

    与上一个中间件不同，这里不保留第一条消息。
    如果截断后 AI 不记得用户名字 Bob，就证明中间件确实生效了。

    Args:
        state: 当前 Agent 状态
        runtime: 运行时上下文

    Returns:
        返回仅包含最后两条消息的字典，用于替换完整消息历史
    """
    messages = state["messages"]

    return {
        "messages": [
            # 清空所有已有消息
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            # 仅保留最后 2 条消息（最近的一问一答）
            *messages[-2:]
        ]
    }

# 创建 Agent 并注册激进截断中间件
agent = create_agent(
    basic_model,
    middleware=[trim_without_first_message],  # 注册新的截断中间件
    checkpointer=InMemorySaver(),
)

# 执行相同的 4 轮对话测试
agent_invoke(agent)


================================== Ai Message ==================================

I don't have access to information about you personally, including your name. I only know what you choose to share with me in our conversations. If you'd like to tell me your name, I'd be happy to know it! Otherwise, I'll continue to refer to you as "you" or "my friend."


现在智能体不记得我是谁，说明中间件确实生效了！

## 三、敏感词过滤

**护栏**（Guardrails）是智能体提供的一类内容安全能力的统称。大模型本身具备一定的内容风控能力，但很容易被突破。搜索「大模型破甲」就能找到此类教程。智能体可以在模型之外，提供额外的安全保护。这是通过工程上的强制检查实现的。

在 LangGraph 中，护栏可以通过中间件轻松实现。下面我们实现一个简单的护栏：若用户的最新消息中包含某些敏感词，智能体将拒绝回答。

In [7]:
from typing import Any

from langchain.agents.middleware import before_agent, AgentState  # @before_agent：在 Agent 执行前运行
from langgraph.runtime import Runtime

# 定义敏感词黑名单：包含这些词的请求将被拦截
banned_keywords = ["hack", "exploit", "malware"]

@before_agent(can_jump_to=["end"])
def content_filter(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    敏感词过滤中间件：基于关键词的确定性护栏。

    在 Agent 执行前检查用户消息，如果包含黑名单中的敏感词，
    则直接拦截请求，阻止 Agent 继续处理。

    can_jump_to=["end"]: 允许中间件将工作流跳转到 "end" 节点，
    从而提前终止 Agent 的正常执行流程。

    Args:
        state: 当前 Agent 状态
        runtime: 运行时上下文

    Returns:
        如果发现敏感词，返回包含拒绝回复的消息列表 + jump_to="end" 指令
        否则返回 None，允许 Agent 正常执行
    """
    # 空消息列表直接放行
    if not state["messages"]:
        return None

    # 获取最后一条消息（用户最新输入）
    last_message = state["messages"][-1]
    # 只检查用户消息，AI 的回复不需要过滤
    if last_message.type != "human":
        return None

    # 统一转换为小写，避免大小写绕过
    content = last_message.content.lower()

    # 逐词检查是否包含敏感词
    for keyword in banned_keywords:
        if keyword in content:
            # 发现敏感词：拦截请求，返回预设的拒绝回复
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "I cannot process requests containing inappropriate content. Please rephrase your request."
                }],
                # 跳转到 "end" 节点，跳过 Agent 的正常处理流程
                "jump_to": "end"
            }

    # 未发现问题，正常执行
    return None

# 创建 Agent 并注册敏感词过滤中间件
agent = create_agent(
    model=basic_model,
    middleware=[content_filter],
)

# 测试：发送包含敏感词 "hack" 的请求，预期被中间件拦截
result = agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a database?"}]
})


In [8]:
# 打印交互记录：验证用户请求被拦截，AI 返回了预设的拒绝回复
for message in result["messages"]:
    message.pretty_print()


================================ Human Message =================================

How do I hack into a database?
================================== Ai Message ==================================

I cannot process requests containing inappropriate content. Please rephrase your request.


## 四、PII 检测

接下来，我们继续编写护栏。[PII](https://docs.langchain.com/oss/python/langchain/guardrails#pii-detection)（Personally Identifiable Information）检测可以发现用户输入中的邮箱、IP、地址、银行卡等隐私信息，并做出处置。

下面的例子来源于生活。我们经常把报错复制给大模型，让它帮忙 debug。但报错中可能包含个人隐私信息。针对这种情况，采用以下两种方法进行处置：

1. 拒绝回答问题
2. 屏蔽隐私信息

In [9]:
from textwrap import dedent  # dedent 用于去除多行字符串的缩进
from pydantic import BaseModel, Field  # 用于定义结构化输出格式

# 可信模型：用于 PII 检测的判断，一般使用本地模型或可信模型
# 这里仍使用 DashScope 的 qwen3-coder-plus 作为示例
trusted_model = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    model="qwen3-coder-plus",
)

# Pydantic 模型：定义 PII 检测的结构化输出格式
# with_structured_output 会将模型的输出约束为此格式
class PiiCheck(BaseModel):
    """检测文本是否包含个人可识别信息（PII）的结构化输出。"""
    is_pii: bool = Field(description="Whether the text contains PII (个人可识别信息)")

def message_with_pii(pii_middleware):
    """
    辅助函数：用给定的 PII 检测中间件创建 Agent 并执行测试。

    测试用例：一条包含文件路径（含用户名 luochang）的报错信息。
    这种场景很常见 —— 开发者常把报错日志粘贴给 AI 调试，但日志中可能泄露隐私。

    Args:
        pii_middleware: PII 检测中间件函数

    Returns:
        Agent 执行结果
    """
    agent = create_agent(
        model=basic_model,  # 主 Agent 使用低费率模型
        middleware=[pii_middleware],  # 注册 PII 检测中间件
    )

    # 发送包含文件路径（含用户名）的测试请求
    # 文件路径中的 "luochang" 是用户名，应被识别为 PII
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": dedent(
                """
                File "/home/luochang/proj/agent.py", line 53, in my_agent
                    agent = create_react_agent(
                ---
                报错位置在哪
                """).strip()  # 去除多行字符串首尾的空白
        }]
    })

    return result


🍉 **处置方式一**：如遇隐私信息，拒绝回复。

In [10]:
@before_agent(can_jump_to=["end"])
def content_blocker(state: AgentState,  runtime: Runtime) -> dict[str, Any] | None:
    """
    PII 检测中间件（处置方式一）：发现隐私信息时拒绝回答。

    工作流程：
    1. 提取用户最新消息
    2. 调用可信模型判断是否包含 PII
    3. 若包含 PII → 拒绝回答，终止流程
    4. 若不包含 PII → 放行，允许 Agent 正常执行
    """
    # 空消息列表直接放行
    if not state["messages"]:
        return None

    # 只处理用户消息
    last_message = state["messages"][-1]
    if last_message.type != "human":
        return None

    content = last_message.content.lower()

    # 构造 PII 检测提示词：要求模型判断文本中是否包含个人可识别信息
    prompt = (
        "你是一个隐私保护助手。请识别下面文本中涉及个人可识别信息（PII），"
        "例如：姓名、身份证号、护照号、电话号码、邮箱、住址、银行卡号、社交账号、车牌等。"
        "特别注意，若代码、文件路径中包含用户名，也应被视为敏感信息。"
        "若包含敏感信息，请返回{\"is_pii\": True}，否则返回{\"is_pii\": False}。"
        "请严格以 json 格式返回，并且只输出 json。文本如下：\n\n" + content
    )

    # 使用可信模型 + 结构化输出，确保返回格式符合 PiiCheck 模型定义
    pii_agent = trusted_model.with_structured_output(PiiCheck)
    result = pii_agent.invoke(prompt)

    # 根据检测结果决定下一步操作
    if result.is_pii is True:
        # 发现 PII：拒绝回答，直接终止流程
        return {
            "messages": [{
                "role": "assistant",
                "content": "I cannot process requests containing inappropriate content. Please rephrase your request."
            }],
            "jump_to": "end"  # 跳转到 end 节点，跳过 Agent 的正常处理
        }
    else:
        # 未发现 PII，打印日志并放行
        print("No PII found")

    return None


In [11]:
# 测试 PII 检测中间件（拒绝回答方式）
# 预期：由于文件路径中包含用户名 "luochang"，被识别为 PII，AI 拒绝回答
result = message_with_pii(pii_middleware=content_blocker)

# 打印交互记录，验证拦截效果
for message in result["messages"]:
    message.pretty_print()


================================ Human Message =================================

File "/home/luochang/proj/agent.py", line 53, in my_agent
    agent = create_react_agent(
---
报错位置在哪
================================== Ai Message ==================================

I cannot process requests containing inappropriate content. Please rephrase your request.


🏀 **处置方式二**：如遇敏感信息，使用一串  `*****` 号屏蔽隐私信息。

In [12]:
@before_agent(can_jump_to=["end"])
def content_filter(state: AgentState,  runtime: Runtime) -> dict[str, Any] | None:
    """
    PII 检测中间件（处置方式二）：发现隐私信息时用星号屏蔽。

    工作流程：
    1. 提取用户最新消息
    2. 调用可信模型判断是否包含 PII
    3. 若包含 PII → 调用模型对敏感信息打码（替换为 *），将处理后的消息作为 AI 回复
    4. 若不包含 PII → 放行，允许 Agent 正常执行
    """
    # 空消息列表直接放行
    if not state["messages"]:
        return None

    # 只处理用户消息
    last_message = state["messages"][-1]
    if last_message.type != "human":
        return None

    content = last_message.content.lower()

    # 第一步：检测是否包含 PII
    prompt = (
        "你是一个隐私保护助手。请识别下面文本中涉及个人可识别信息（PII），"
        "例如：姓名、身份证号、护照号、电话号码、邮箱、住址、银行卡号、社交账号、车牌等。"
        "特别注意，若代码、文件路径中包含用户名，也应被视为敏感信息。"
        "若包含敏感信息，请返回{\"is_pii\": True}，否则返回{\"is_pii\": False}。"
        "请严格以 json 格式返回，并且只输出 json。文本如下：\n\n" + content
    )

    pii_agent = trusted_model.with_structured_output(PiiCheck)
    result = pii_agent.invoke(prompt)

    if result.is_pii is True:
        # 第二步：发现 PII，构造打码提示词，让模型用星号替换敏感信息
        mask_prompt = (
            "你是一个隐私保护助手。请将下面文本中的所有个人可识别信息（PII）用星号（*）替换。"
            "仅替换敏感片段，其他文本保持不变。"
            "只输出处理后的文本，不要任何解释或额外内容。文本如下：\n\n" + last_message.content
        )
        # 调用模型对原始消息进行打码处理
        masked_message = basic_model.invoke(mask_prompt)

        # 将打码后的消息作为 AI 回复返回，同时跳转到 end 终止后续处理
        return {
            "messages": [{
                "role": "assistant",
                "content": masked_message.content  # 模型返回的打码后文本
            }]
        }
    else:
        # 未发现 PII，打印日志并放行
        print("No PII found")

    return None


In [13]:
# 测试 PII 检测中间件（星号屏蔽方式）
# 预期：文件路径中的 "luochang" 被识别为 PII，然后被星号替换为 "/home/*/*/proj/agent.py"
result = message_with_pii(pii_middleware=content_filter)

# 打印交互记录：第一行是用户原始输入，第二行是打码后的消息
for message in result["messages"]:
    message.pretty_print()


================================ Human Message =================================

File "/home/luochang/proj/agent.py", line 53, in my_agent
    agent = create_react_agent(
---
报错位置在哪
================================== Ai Message ==================================

File "/home/*/*/proj/agent.py", line 53, in my_agent
    agent = create_react_agent(
---
报错位置在哪
================================== Ai Message ==================================

根据你提供的错误信息，报错位置在：

**文件路径**：`/home/luochang/proj/agent.py`
**代码行数**：第53行
**函数名**：`my_agent`
**具体位置**：`create_react_agent(` 这个函数调用处

报错的具体位置是 `create_react_agent(` 函数调用的这一行。

不过你只提供了错误的位置信息，没有提供具体的错误内容。要完全解决这个问题，我需要知道：

1. **完整的错误信息**（包括错误类型和详细描述）
2. **第53行附近的完整代码**

常见的可能原因包括：
- 参数传递错误
- 导入模块问题
- 依赖库版本不兼容
- 变量未定义等

如果你能提供完整的错误信息和相关代码片段，我可以给出更准确的解决方案。
